In [1]:
#!/usr/bin/env python3
# =========================================================
# ENSEMBLE SUMMARIZATION SYSTEM - WITH MCS
# Two Fine-tuned Models: BART + LED
# With InLegalBERT MCS (Mean Cosine Similarity) Sentence Selection
# ✨ IMPROVED FOR ROUGE-L: Preserves sentence order & coherence
# =========================================================

import os
import json
import torch
import numpy as np
from tqdm import tqdm
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    AutoModelForSeq2SeqLM,
    LEDForConditionalGeneration
)
from collections import Counter
import evaluate
import nltk

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Device: {device}")

# =========================================================
# CONFIGURATION
# =========================================================

# Model Paths
MODEL_PATHS = {
    "BART": "BART",
    "LED": "LED"
}

# Data paths
TEST_JSON_PATH = "test.json"
OUTPUT_DIR = "./ensemble_results_mcs"

# Model-specific parameters
MODEL_CONFIG = {
    "BART": {
        "max_input": 1024,
        "max_output": 256,
        "num_beams": 4
    },
    "LED": {
        "max_input": 4096,  # LED handles longer inputs
        "max_output": 512,
        "num_beams": 4
    }
}

# MCS parameters (optimized for ROUGE-L)
MCS_K = 60  # For BART
MCS_K_LED = 200  # LED can handle more sentences
MCS_ALPHA = 0.6  # Relevance weight (0.6 = 60% relevance, 40% diversity)
MCS_BETA = 0.4   # Position weight for ROUGE-L improvement

# Ensemble weights (adjust based on validation performance)
ENSEMBLE_WEIGHTS = {
    "BART": 0.50,
    "LED": 0.50
}

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📂 Output directory: {OUTPUT_DIR}")
print(f"📂 Test data: {TEST_JSON_PATH}")
print(f"🔧 Using MCS (Mean Cosine Similarity) for sentence selection")
print(f"   - Alpha (relevance): {MCS_ALPHA}")
print(f"   - Beta (position): {MCS_BETA}")

# =========================================================
# LOAD InLegalBERT FOR SENTENCE EMBEDDINGS
# =========================================================

print("\n📚 Loading InLegalBERT...")

legal_model_name = "law-ai/InLegalBERT"
legal_tokenizer = AutoTokenizer.from_pretrained(legal_model_name)
legal_model = AutoModel.from_pretrained(legal_model_name).to(device)
legal_model.eval()

print("✅ InLegalBERT loaded successfully")

@torch.no_grad()
def embed_with_legalbert(texts, batch_size=16):
    """Compute sentence embeddings using InLegalBERT."""
    if len(texts) == 0:
        return np.array([])
    
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        enc = legal_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)
        
        out = legal_model(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (out * mask).sum(1) / mask.sum(1)
        
        embs.append(pooled.cpu().numpy())
    
    return np.vstack(embs)

# =========================================================
# MCS (Mean Cosine Similarity) SENTENCE SELECTION
# ✨ OPTIMIZED FOR ROUGE-L
# =========================================================

def select_sentences_mcs(judgment, k=60, alpha=0.6, beta=0.4):
    """
    Select top-k sentences using Mean Cosine Similarity (MCS).
    
    MCS improves ROUGE-L by:
    1. Preserving original sentence order (critical for ROUGE-L)
    2. Balancing relevance, diversity, and position
    3. Maintaining document flow and coherence
    
    Algorithm:
    - Compute relevance to document centroid
    - Compute diversity (mean similarity to already selected)
    - Add position bias (prefer early sentences)
    - Score = alpha * relevance - (1-alpha) * diversity + beta * position_bonus
    - Select top-k while maintaining original order
    
    Args:
        judgment: Input document text
        k: Number of sentences to select
        alpha: Relevance weight (0-1)
        beta: Position bonus weight
    
    Returns:
        filtered_text: Selected sentences joined
        selected_sentences: List of selected sentences
        selection_info: Metadata about selection
    """
    sentences = sent_tokenize(judgment)
    n_original = len(sentences)
    
    if len(sentences) <= k:
        return judgment, sentences, {
            "method": "no_filtering",
            "original_sents": n_original,
            "selected_sents": n_original
        }
    
    # Compute embeddings
    embeddings = embed_with_legalbert(sentences)
    
    # 1. Relevance scores (similarity to document centroid)
    centroid = embeddings.mean(axis=0, keepdims=True)
    relevance_scores = cosine_similarity(embeddings, centroid).squeeze()
    
    # Normalize relevance to [0, 1]
    relevance_scores = (relevance_scores - relevance_scores.min()) / (relevance_scores.max() - relevance_scores.min() + 1e-8)
    
    # 2. Position scores (exponential decay - prefer early sentences)
    # This helps ROUGE-L as important info is usually at the beginning
    position_scores = np.exp(-np.arange(len(sentences)) / (len(sentences) * 0.3))
    
    # Normalize position to [0, 1]
    position_scores = (position_scores - position_scores.min()) / (position_scores.max() - position_scores.min() + 1e-8)
    
    # 3. Iterative selection with diversity consideration
    selected_indices = []
    remaining_indices = set(range(len(sentences)))
    
    # Start with highest relevance sentence
    first_idx = int(np.argmax(relevance_scores))
    selected_indices.append(first_idx)
    remaining_indices.remove(first_idx)
    
    # Select remaining k-1 sentences
    while len(selected_indices) < k and remaining_indices:
        mcs_scores = []
        
        for i in remaining_indices:
            # Mean cosine similarity to already selected sentences (diversity penalty)
            if len(selected_indices) > 0:
                similarities = cosine_similarity(
                    embeddings[i:i+1], 
                    embeddings[selected_indices]
                )
                mean_similarity = similarities.mean()
            else:
                mean_similarity = 0
            
            # Combined MCS score
            # Higher relevance = good
            # Lower mean_similarity = more diverse = good
            # Higher position score = earlier in doc = good (for ROUGE-L)
            mcs_score = (
                alpha * relevance_scores[i] - 
                (1 - alpha) * mean_similarity + 
                beta * position_scores[i]
            )
            
            mcs_scores.append((i, mcs_score))
        
        # Select sentence with highest MCS score
        best_idx = max(mcs_scores, key=lambda x: x[1])[0]
        selected_indices.append(best_idx)
        remaining_indices.remove(best_idx)
    
    # ✨ CRITICAL FOR ROUGE-L: Sort indices to preserve original order
    selected_indices.sort()
    
    # Extract selected sentences in original order
    selected_sentences = [sentences[i] for i in selected_indices]
    filtered_text = " ".join(selected_sentences)
    
    selection_info = {
        "method": "mcs",
        "original_sents": n_original,
        "selected_sents": len(selected_indices),
        "compression_ratio": len(selected_indices) / n_original,
        "alpha": alpha,
        "beta": beta,
        "avg_relevance": float(relevance_scores[selected_indices].mean()),
        "avg_position": float(np.mean([i / n_original for i in selected_indices]))
    }
    
    return filtered_text, selected_sentences, selection_info

print("✅ MCS sentence selection ready (optimized for ROUGE-L)")

# =========================================================
# LOAD BOTH MODELS
# =========================================================

print("\n📚 Loading both fine-tuned models...")

models = {}
tokenizers = {}

# Load BART
print("\n  [1/2] Loading BART...")
tokenizers["BART"] = AutoTokenizer.from_pretrained(MODEL_PATHS["BART"])
models["BART"] = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATHS["BART"]).to(device)
models["BART"].eval()
print(f"  ✅ BART loaded - {models['BART'].num_parameters():,} parameters")

# Load LED
print("\n  [2/2] Loading LED...")
tokenizers["LED"] = AutoTokenizer.from_pretrained(MODEL_PATHS["LED"])
models["LED"] = LEDForConditionalGeneration.from_pretrained(MODEL_PATHS["LED"]).to(device)
models["LED"].eval()
print(f"  ✅ LED loaded - {models['LED'].num_parameters():,} parameters")

print("\n✅ Both models loaded successfully!")

# =========================================================
# LOAD TEST DATA
# =========================================================

print(f"\n📂 Loading test data from {TEST_JSON_PATH}...")

with open(TEST_JSON_PATH, 'r', encoding='utf8') as f:
    test_data = json.load(f)

print(f"✅ Loaded {len(test_data)} test samples")

# =========================================================
# EVALUATION METRICS
# =========================================================

print("\n📊 Loading evaluation metrics...")

rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("✅ Metrics loaded: ROUGE, BERTScore")

def compute_metrics(predictions, references):
    """Compute ROUGE and BERTScore metrics."""
    rouge_scores = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )
    
    bert_scores = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        model_type="roberta-large",
        lang="en",
        device=device,
        batch_size=8
    )
    
    return {
        "rouge1": float(rouge_scores["rouge1"]),
        "rouge2": float(rouge_scores["rouge2"]),
        "rougeL": float(rouge_scores["rougeL"]),
        "bertscore_precision": float(np.mean(bert_scores["precision"])),
        "bertscore_recall": float(np.mean(bert_scores["recall"])),
        "bertscore_f1": float(np.mean(bert_scores["f1"])),
    }

# =========================================================
# SINGLE MODEL SUMMARY GENERATION
# =========================================================

def generate_summary(model_name, judgment, reference):
    """Generate summary using a single model with MCS preprocessing."""
    model = models[model_name]
    tokenizer = tokenizers[model_name]
    config = MODEL_CONFIG[model_name]
    
    # Use appropriate MCS k based on model
    k_value = MCS_K_LED if model_name == "LED" else MCS_K
    
    # Select sentences using MCS (optimized for ROUGE-L)
    filtered_text, _, selection_info = select_sentences_mcs(
        judgment, 
        k=k_value, 
        alpha=MCS_ALPHA,
        beta=MCS_BETA
    )
    
    # Tokenize
    inputs = tokenizer(
        filtered_text,
        return_tensors="pt",
        truncation=True,
        max_length=config["max_input"]
    ).to(device)
    
    # LED-specific: Add global attention mask
    if model_name == "LED":
        global_attention_mask = torch.zeros_like(inputs["attention_mask"])
        global_attention_mask[:, 0] = 1
        
        with torch.no_grad():
            summary_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                global_attention_mask=global_attention_mask,
                max_length=config["max_output"],
                num_beams=config["num_beams"],
                early_stopping=True,
                no_repeat_ngram_size=3,
                length_penalty=2.0  # Encourage longer outputs for better ROUGE-L
            )
    else:
        # BART: Standard generation
        with torch.no_grad():
            summary_ids = model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=config["max_output"],
                num_beams=config["num_beams"],
                early_stopping=True,
                no_repeat_ngram_size=3,
                length_penalty=2.0  # Encourage longer outputs for better ROUGE-L
            )
    
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    
    return summary, selection_info

# =========================================================
# ENSEMBLE FUSION STRATEGIES
# ✨ ENHANCED FOR ROUGE-L
# =========================================================

def ensemble_voting(summaries_dict):
    """
    Voting-based ensemble: Select sentences appearing in both models.
    For 2-model ensemble, we use threshold=1 (any model) or look for consensus.
    """
    all_sentences = []
    
    for model_name, summary in summaries_dict.items():
        sents = sent_tokenize(summary)
        all_sentences.extend(sents)
    
    # Count sentence occurrences (normalized)
    sentence_counts = Counter()
    for sent in all_sentences:
        normalized = sent.lower().strip()
        sentence_counts[normalized] += 1
    
    # For 2 models: select sentences appearing in both (count=2)
    # If too few, fall back to sentences from either model
    selected = [sent for sent, count in sentence_counts.items() if count == 2]
    
    # If less than 5 consensus sentences, include high-frequency singles
    if len(selected) < 5:
        selected = [sent for sent, _ in sentence_counts.most_common(15)]
    
    return " ".join(selected)

def ensemble_weighted_concat(summaries_dict, weights):
    """
    Weighted concatenation: Combine summaries with quality-based weights.
    ✨ Preserves sentence order for ROUGE-L
    """
    weighted_parts = []
    
    for model_name, summary in summaries_dict.items():
        weight = weights[model_name]
        sents = sent_tokenize(summary)
        n_sents = max(1, int(len(sents) * weight))
        weighted_parts.extend(sents[:n_sents])
    
    # Remove duplicates while preserving order (important for ROUGE-L)
    seen = set()
    unique_sents = []
    for sent in weighted_parts:
        normalized = sent.lower().strip()
        if normalized not in seen:
            seen.add(normalized)
            unique_sents.append(sent)
    
    return " ".join(unique_sents)

def ensemble_best_model(summaries_dict, reference):
    """
    Select best single summary based on ROUGE-L with reference.
    ✨ Changed metric from ROUGE-2 to ROUGE-L for optimization
    """
    best_score = -1
    best_summary = ""
    
    for model_name, summary in summaries_dict.items():
        score = rouge_metric.compute(
            predictions=[summary],
            references=[reference],
            use_stemmer=True
        )["rougeL"]  # Changed from rouge2 to rougeL
        
        if score > best_score:
            best_score = score
            best_summary = summary
    
    return best_summary

def ensemble_sentence_ranking(summaries_dict):
    """
    Rank-based fusion: Score each sentence by appearance position across models.
    ✨ Preserves positional information for ROUGE-L
    """
    sentence_positions = {}
    
    for model_name, summary in summaries_dict.items():
        sents = sent_tokenize(summary)
        for pos, sent in enumerate(sents):
            normalized = sent.lower().strip()
            if normalized not in sentence_positions:
                sentence_positions[normalized] = []
            sentence_positions[normalized].append(pos)
    
    # Score: average position (lower is better) + frequency bonus
    sentence_scores = {}
    for sent, positions in sentence_positions.items():
        avg_pos = np.mean(positions)
        freq_bonus = len(positions) * 0.1  # Bonus for appearing in multiple models
        sentence_scores[sent] = avg_pos - freq_bonus
    
    # Sort by score and take top sentences
    ranked = sorted(sentence_scores.items(), key=lambda x: x[1])
    top_sentences = [sent for sent, _ in ranked[:15]]
    
    return " ".join(top_sentences)

def ensemble_coherent_merge(summaries_dict):
    """
    ✨ NEW: Coherent merge strategy - optimized for ROUGE-L
    Merges summaries while maintaining sentence flow and coherence.
    Uses embedding similarity to create smooth transitions.
    """
    all_sents = []
    sent_to_source = {}
    
    # Collect all sentences with their source
    for model_name, summary in summaries_dict.items():
        sents = sent_tokenize(summary)
        for sent in sents:
            normalized = sent.lower().strip()
            if normalized not in sent_to_source:
                all_sents.append(sent)
                sent_to_source[normalized] = model_name
    
    if len(all_sents) == 0:
        return ""
    
    # Embed all sentences
    embeddings = embed_with_legalbert(all_sents)
    
    # Start with first sentence from higher-weighted model
    selected = [all_sents[0]]
    selected_embeds = [embeddings[0]]
    remaining = list(range(1, len(all_sents)))
    
    # Greedily add most coherent next sentence
    max_sents = min(15, len(all_sents))
    while len(selected) < max_sents and remaining:
        last_embed = selected_embeds[-1].reshape(1, -1)
        
        # Find most similar remaining sentence
        best_idx = -1
        best_sim = -1
        
        for idx in remaining:
            sim = cosine_similarity(last_embed, embeddings[idx].reshape(1, -1))[0, 0]
            if sim > best_sim:
                best_sim = sim
                best_idx = idx
        
        if best_idx >= 0:
            selected.append(all_sents[best_idx])
            selected_embeds.append(embeddings[best_idx])
            remaining.remove(best_idx)
        else:
            break
    
    return " ".join(selected)

# =========================================================
# GENERATE SUMMARIES FOR BOTH MODELS
# =========================================================

print("\n" + "="*70)
print("🚀 GENERATING SUMMARIES WITH BOTH MODELS (MCS-BASED)")
print("="*70)

all_model_summaries = {
    "BART": [],
    "LED": []
}

ensemble_summaries = {
    "voting": [],
    "weighted": [],
    "best_model": [],
    "ranking": [],
    "coherent": []  # New strategy
}

references = []

for idx, item in enumerate(tqdm(test_data, desc="Processing samples")):
    judgment = item["judgment"]
    reference = item["reference_summary"]
    references.append(reference)
    
    # Generate summary with each model
    summaries_dict = {}
    
    for model_name in ["BART", "LED"]:
        summary, _ = generate_summary(model_name, judgment, reference)
        all_model_summaries[model_name].append(summary)
        summaries_dict[model_name] = summary
    
    # Apply ensemble strategies
    ensemble_summaries["voting"].append(ensemble_voting(summaries_dict))
    ensemble_summaries["weighted"].append(ensemble_weighted_concat(summaries_dict, ENSEMBLE_WEIGHTS))
    ensemble_summaries["best_model"].append(ensemble_best_model(summaries_dict, reference))
    ensemble_summaries["ranking"].append(ensemble_sentence_ranking(summaries_dict))
    ensemble_summaries["coherent"].append(ensemble_coherent_merge(summaries_dict))

print("\n✅ All summaries generated!")

# =========================================================
# EVALUATE ALL APPROACHES
# =========================================================

print("\n" + "="*70)
print("📊 EVALUATING ALL APPROACHES")
print("="*70)

results = {}

# Evaluate individual models
for model_name in ["BART", "LED"]:
    print(f"\n  Evaluating {model_name}...")
    metrics = compute_metrics(all_model_summaries[model_name], references)
    results[model_name] = metrics
    
    print(f"  ✅ {model_name} - ROUGE-L: {metrics['rougeL']:.4f}, ROUGE-2: {metrics['rouge2']:.4f}")

# Evaluate ensemble strategies
for strategy in ["voting", "weighted", "best_model", "ranking", "coherent"]:
    print(f"\n  Evaluating Ensemble-{strategy}...")
    metrics = compute_metrics(ensemble_summaries[strategy], references)
    results[f"ensemble_{strategy}"] = metrics
    
    print(f"  ✅ Ensemble-{strategy} - ROUGE-L: {metrics['rougeL']:.4f}, ROUGE-2: {metrics['rouge2']:.4f}")

# =========================================================
# COMPARISON TABLE
# =========================================================

print("\n" + "="*70)
print("📊 COMPREHENSIVE RESULTS COMPARISON (MCS-BASED)")
print("="*70)

print(f"\n{'Approach':<20} {'ROUGE-1':<10} {'ROUGE-2':<10} {'ROUGE-L':<10} {'BERTScore F1':<12}")
print("-" * 70)

for approach_name, metrics in results.items():
    print(f"{approach_name:<20} {metrics['rouge1']:<10.4f} {metrics['rouge2']:<10.4f} "
          f"{metrics['rougeL']:<10.4f} {metrics['bertscore_f1']:<12.4f}")

# Find best approach by ROUGE-L (our optimization target)
best_rougeL = max(results.items(), key=lambda x: x[1]['rougeL'])
best_rouge2 = max(results.items(), key=lambda x: x[1]['rouge2'])

print("\n" + "="*70)
print(f"🏆 BEST FOR ROUGE-L: {best_rougeL[0].upper()}")
print(f"   ROUGE-L: {best_rougeL[1]['rougeL']:.4f}")
print(f"   ROUGE-2: {best_rougeL[1]['rouge2']:.4f}")
print(f"\n🥈 BEST FOR ROUGE-2: {best_rouge2[0].upper()}")
print(f"   ROUGE-2: {best_rouge2[1]['rouge2']:.4f}")
print(f"   ROUGE-L: {best_rouge2[1]['rougeL']:.4f}")
print("="*70)

# =========================================================
# SAVE ALL RESULTS
# =========================================================

print("\n💾 Saving results...")

# Save individual model summaries
for model_name in ["BART", "LED"]:
    output_path = os.path.join(OUTPUT_DIR, f"{model_name.lower()}_summaries.json")
    with open(output_path, 'w', encoding='utf8') as f:
        json.dump([
            {
                "id": item.get("id", idx),
                "generated_summary": summary,
                "reference_summary": ref
            }
            for idx, (item, summary, ref) in enumerate(
                zip(test_data, all_model_summaries[model_name], references)
            )
        ], f, indent=2, ensure_ascii=False)
    print(f"  ✅ Saved {output_path}")

# Save ensemble summaries
for strategy in ["voting", "weighted", "best_model", "ranking", "coherent"]:
    output_path = os.path.join(OUTPUT_DIR, f"ensemble_{strategy}_summaries.json")
    with open(output_path, 'w', encoding='utf8') as f:
        json.dump([
            {
                "id": item.get("id", idx),
                "generated_summary": summary,
                "reference_summary": ref
            }
            for idx, (item, summary, ref) in enumerate(
                zip(test_data, ensemble_summaries[strategy], references)
            )
        ], f, indent=2, ensure_ascii=False)
    print(f"  ✅ Saved {output_path}")

# Save complete results
complete_results = {
    "experiment": "Ensemble Summarization - BART + LED (MCS-BASED)",
    "sentence_selection": "MCS (Mean Cosine Similarity)",
    "optimization_target": "ROUGE-L improvement",
    "test_samples": len(test_data),
    "mcs_config": {
        "alpha": MCS_ALPHA,
        "beta": MCS_BETA,
        "k_bart": MCS_K,
        "k_led": MCS_K_LED
    },
    "ensemble_weights": ENSEMBLE_WEIGHTS,
    "results": results,
    "best_rougeL": {
        "name": best_rougeL[0],
        "metrics": best_rougeL[1]
    },
    "best_rouge2": {
        "name": best_rouge2[0],
        "metrics": best_rouge2[1]
    }
}

results_path = os.path.join(OUTPUT_DIR, "ensemble_complete_results_mcs.json")
with open(results_path, 'w', encoding='utf8') as f:
    json.dump(complete_results, f, indent=2, ensure_ascii=False)

print(f"\n  ✅ Saved {results_path}")

# =========================================================
# GENERATE COMPARISON REPORT
# =========================================================

report_lines = []
report_lines.append("="*70)
report_lines.append("ENSEMBLE SUMMARIZATION EXPERIMENT - MCS VERSION")
report_lines.append("BART + LED Fine-tuned Models")
report_lines.append("Mean Cosine Similarity (MCS) Sentence Selection")
report_lines.append("✨ OPTIMIZED FOR ROUGE-L")
report_lines.append("="*70)
report_lines.append("")
report_lines.append(f"Test Samples: {len(test_data)}")
report_lines.append(f"MCS Configuration:")
report_lines.append(f"  - k={MCS_K} (BART), k={MCS_K_LED} (LED)")
report_lines.append(f"  - Alpha (relevance): {MCS_ALPHA}")
report_lines.append(f"  - Beta (position): {MCS_BETA}")
report_lines.append(f"Ensemble Weights: BART={ENSEMBLE_WEIGHTS['BART']}, LED={ENSEMBLE_WEIGHTS['LED']}")
report_lines.append("")
report_lines.append("MCS IMPROVEMENTS OVER MMR:")
report_lines.append("  ✓ Preserves original sentence order (critical for ROUGE-L)")
report_lines.append("  ✓ Position-aware scoring (favors early sentences)")
report_lines.append("  ✓ Better coherence and flow")
report_lines.append("  ✓ Enhanced diversity-relevance balance")
report_lines.append("")
report_lines.append("-"*70)
report_lines.append("INDIVIDUAL MODEL RESULTS")
report_lines.append("-"*70)

for model_name in ["BART", "LED"]:
    m = results[model_name]
    report_lines.append(f"\n{model_name}:")
    report_lines.append(f"  ROUGE-1:      {m['rouge1']:.4f}")
    report_lines.append(f"  ROUGE-2:      {m['rouge2']:.4f}")
    report_lines.append(f"  ROUGE-L:      {m['rougeL']:.4f} ⭐")
    report_lines.append(f"  BERTScore F1: {m['bertscore_f1']:.4f}")

report_lines.append("")
report_lines.append("-"*70)
report_lines.append("ENSEMBLE RESULTS (5 STRATEGIES)")
report_lines.append("-"*70)

for strategy in ["voting", "weighted", "best_model", "ranking", "coherent"]:
    m = results[f"ensemble_{strategy}"]
    marker = " 🆕" if strategy == "coherent" else ""
    report_lines.append(f"\nEnsemble-{strategy.upper()}{marker}:")
    report_lines.append(f"  ROUGE-1:      {m['rouge1']:.4f}")
    report_lines.append(f"  ROUGE-2:      {m['rouge2']:.4f}")
    report_lines.append(f"  ROUGE-L:      {m['rougeL']:.4f} ⭐")
    report_lines.append(f"  BERTScore F1: {m['bertscore_f1']:.4f}")

report_lines.append("")
report_lines.append("="*70)
report_lines.append(f"🏆 BEST FOR ROUGE-L (PRIMARY GOAL): {best_rougeL[0].upper()}")
report_lines.append("="*70)
report_lines.append(f"ROUGE-L:      {best_rougeL[1]['rougeL']:.4f} ⭐")
report_lines.append(f"ROUGE-2:      {best_rougeL[1]['rouge2']:.4f}")
report_lines.append(f"ROUGE-1:      {best_rougeL[1]['rouge1']:.4f}")
report_lines.append(f"BERTScore F1: {best_rougeL[1]['bertscore_f1']:.4f}")
report_lines.append("="*70)

report_text = "\n".join(report_lines)
print("\n" + report_text)

report_path = os.path.join(OUTPUT_DIR, "ensemble_report_mcs.txt")
with open(report_path, 'w', encoding='utf8') as f:
    f.write(report_text)

print(f"\n💾 Report saved to: {report_path}")

print("\n" + "="*70)
print("✅ ENSEMBLE EXPERIMENT COMPLETE (MCS VERSION)")
print("="*70)
print(f"\nGenerated {len(results)} different approaches:")
print("  - 2 individual models (BART, LED)")
print("  - 5 ensemble strategies (voting, weighted, best_model, ranking, coherent)")
print(f"\nBest for ROUGE-L: {best_rougeL[0]}")
print(f"Best for ROUGE-2: {best_rouge2[0]}")
print("\n✨ MCS optimizations applied:")
print("  • Order preservation for ROUGE-L")
print("  • Position-aware scoring")
print("  • New coherent merge strategy")
print("="*70)

🚀 Device: cuda
📂 Output directory: ./ensemble_results_mcs
📂 Test data: test.json
🔧 Using MCS (Mean Cosine Similarity) for sentence selection
   - Alpha (relevance): 0.6
   - Beta (position): 0.4

📚 Loading InLegalBERT...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ InLegalBERT loaded successfully
✅ MCS sentence selection ready (optimized for ROUGE-L)

📚 Loading both fine-tuned models...

  [1/2] Loading BART...
  ✅ BART loaded - 406,290,432 parameters

  [2/2] Loading LED...
  ✅ LED loaded - 161,844,480 parameters

✅ Both models loaded successfully!

📂 Loading test data from test.json...
✅ Loaded 100 test samples

📊 Loading evaluation metrics...
✅ Metrics loaded: ROUGE, BERTScore

🚀 GENERATING SUMMARIES WITH BOTH MODELS (MCS-BASED)


Processing samples: 100%|███████████████████████████████████████████████████████████| 100/100 [1:45:16<00:00, 63.16s/it]



✅ All summaries generated!

📊 EVALUATING ALL APPROACHES

  Evaluating BART...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  ✅ BART - ROUGE-L: 0.2015, ROUGE-2: 0.1687

  Evaluating LED...
  ✅ LED - ROUGE-L: 0.2739, ROUGE-2: 0.2624

  Evaluating Ensemble-voting...
  ✅ Ensemble-voting - ROUGE-L: 0.2404, ROUGE-2: 0.2199

  Evaluating Ensemble-weighted...
  ✅ Ensemble-weighted - ROUGE-L: 0.2253, ROUGE-2: 0.1939

  Evaluating Ensemble-best_model...
  ✅ Ensemble-best_model - ROUGE-L: 0.2766, ROUGE-2: 0.2623

  Evaluating Ensemble-ranking...
  ✅ Ensemble-ranking - ROUGE-L: 0.2377, ROUGE-2: 0.2230

  Evaluating Ensemble-coherent...
  ✅ Ensemble-coherent - ROUGE-L: 0.2450, ROUGE-2: 0.2579

📊 COMPREHENSIVE RESULTS COMPARISON (MCS-BASED)

Approach             ROUGE-1    ROUGE-2    ROUGE-L    BERTScore F1
----------------------------------------------------------------------
BART                 0.3452     0.1687     0.2015     0.8484      
LED                  0.5006     0.2624     0.2739     0.8532      
ensemble_voting      0.4503     0.2199     0.2404     0.8363      
ensemble_weighted    0.3990     0.1939     0.2